In [1]:
# ============================================================================
# SAURNET: Complete Solar Grid Monitoring & Analysis System
# Combined implementation with simulation, CSV analysis, and ML capabilities
# ============================================================================

# Install dependencies (uncomment for Google Colab)
#!pip install gradio pandas numpy scikit-learn matplotlib

import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

C:\Users\BIT\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# CONFIGURATION
# ============================================================================
DAYS = 30
RATED_PANEL_POWER = 5.0  # kW
DROP_THRESHOLD = 0.35    # 35% sudden drop threshold


In [3]:
# ============================================================================
# 1. SYNTHETIC DATA GENERATOR
# ============================================================================
def generate_solar_data(samples=200):
    """Generate synthetic solar panel sensor data"""
    np.random.seed(42)
    time = pd.date_range(start='2025-01-01 09:00', periods=samples, freq='5min')

    irradiance = np.clip(np.random.normal(700, 150, samples), 0, 1000)
    temperature = np.clip(np.random.normal(35, 5, samples), 20, 60)
    voltage = np.clip(30 + irradiance * 0.02 + np.random.normal(0, 1, samples), 0, None)
    current = np.clip(5 + irradiance * 0.01 + np.random.normal(0, 0.5, samples), 0, None)
    power = voltage * current

    data = pd.DataFrame({
        'Timestamp': time,
        'Irradiance': irradiance,
        'Temperature': temperature,
        'Voltage': voltage,
        'Current': current,
        'Power': power
    })
    return data

In [4]:
# ============================================================================
# 2. ML MODELS FOR ANOMALY DETECTION & FORECASTING
# ============================================================================
scaler = MinMaxScaler()
anomaly_model = IsolationForest(contamination=0.05, random_state=42)
forecast_model = RandomForestRegressor(n_estimators=100, random_state=42)

def train_models(data):
    """Train anomaly detection and power forecasting models"""
    features = data[['Irradiance', 'Temperature', 'Voltage', 'Current', 'Power']]
    X_scaled = scaler.fit_transform(features)

    # Train Anomaly Detection
    anomaly_model.fit(X_scaled)

    # Train Forecasting Model
    X_forecast = data[['Irradiance', 'Temperature']]
    y_forecast = data['Power']
    forecast_model.fit(X_forecast, y_forecast)

    return "Models trained successfully"

def detect_anomalies(data):
    """Detect anomalies in solar panel data"""
    data_copy = data.copy()
    features = data_copy[['Irradiance', 'Temperature', 'Voltage', 'Current', 'Power']]
    X_scaled = scaler.transform(features)
    predictions = anomaly_model.predict(X_scaled)

    # -1 for anomalies, 1 for normal
    anomaly_count = np.sum(predictions == -1)
    data_copy['Anomaly'] = predictions

    return data_copy, f"Detected {anomaly_count} anomalies out of {len(data_copy)} samples"

In [5]:
# ============================================================================
# 3. SOLAR SYSTEM SIMULATION (30-DAY)
# ============================================================================
def simulate_system(panel_kw, battery_kwh, efficiency):
    """Simulate 30-day solar system with battery and grid export"""
    # No fixed seed - generates different data each time

    sunlight_hours = np.random.uniform(4, 7, DAYS)
    daily_load = np.random.uniform(8, 15, DAYS)  # kWh consumption
    battery_charge = 0

    generated = []
    grid_sent = []
    battery_state = []

    for day in range(DAYS):
        gen = panel_kw * sunlight_hours[day] * efficiency
        generated.append(gen)

        if gen > daily_load[day]:
            excess = gen - daily_load[day]
            chargeable = min(excess, battery_kwh - battery_charge)
            battery_charge += chargeable
            grid_export = excess - chargeable
        else:
            deficit = daily_load[day] - gen
            discharge = min(deficit, battery_charge)
            battery_charge -= discharge
            grid_export = 0

        grid_sent.append(grid_export)
        battery_state.append(battery_charge)

    df = pd.DataFrame({
        "Day": np.arange(1, DAYS + 1),
        "Solar Generated (kWh)": generated,
        "Grid Exported (kWh)": grid_sent,
        "Battery Level (kWh)": battery_state
    })

    # Efficiency Decision
    avg_gen = np.mean(generated)
    avg_export = np.mean(grid_sent)

    if avg_gen > 20 and avg_export > 3:
        status = "✅ Fully Efficient System"
    elif avg_gen > 15:
        status = "⚠️ Partially Efficient System"
    else:
        status = "❌ Inefficient System"

    return df, status

def run_simulation(panel_kw, battery_kwh, efficiency):
    """Run simulation and create visualization"""
    df, status = simulate_system(panel_kw, battery_kwh, efficiency)

    # Plot
    plt.figure(figsize=(10, 5))
    plt.plot(df["Day"], df["Solar Generated (kWh)"], label="Solar Generation", marker='o')
    plt.plot(df["Day"], df["Grid Exported (kWh)"], label="Grid Export", marker='s')
    plt.plot(df["Day"], df["Battery Level (kWh)"], label="Battery Level", marker='^')
    plt.xlabel("Day")
    plt.ylabel("Energy (kWh)")
    plt.title("Monthly Solar Grid Simulation")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    sim_fig = plt.gcf()
    plt.close()

    return df, status, sim_fig

In [6]:
# ============================================================================
# 4. CSV DATA ANALYZER
# ============================================================================
def analyze_solar_data(file, url):
    """Analyze uploaded CSV solar panel data or from URL"""
    try:
        # Determine data source
        if url and url.strip():
            # Read from URL
            df = pd.read_csv(url.strip())
        elif file is not None:
            # Read from uploaded file
            df = pd.read_csv(file.name)
        else:
            return "Please provide either a CSV file or a URL", None, None, None, None

        df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
        df['date'] = df['DATE_TIME'].dt.date
        df['hour'] = df['DATE_TIME'].dt.hour

        # Daily Aggregation
        daily_dc_power = df.groupby('date')['DC_POWER'].sum()
        daily_ac_power = df.groupby('date')['AC_POWER'].sum()
        daily_yield = df.groupby('date')['DAILY_YIELD'].max()  # Max because it's cumulative per day

        # Calculate efficiency (AC/DC ratio)
        total_dc = df['DC_POWER'].sum()
        total_ac = df['AC_POWER'].sum()
        conversion_efficiency = (total_ac / total_dc * 100) if total_dc > 0 else 0

        # Expected efficiency range for solar inverters is 95-98%
        if conversion_efficiency >= 95:
            system_status = "Fully Efficient ✅"
        elif conversion_efficiency >= 90:
            system_status = "Good Performance ⚠️"
        else:
            system_status = "Underperforming ❌"

        # Sudden Drop Detection (using AC_POWER)
        daily_avg_ac = df.groupby('date')['AC_POWER'].mean()
        alerts = []
        for i in range(1, len(daily_avg_ac)):
            if daily_avg_ac.iloc[i-1] > 0:  # Avoid division by zero
                drop = (daily_avg_ac.iloc[i-1] - daily_avg_ac.iloc[i]) / daily_avg_ac.iloc[i-1]
                if drop > DROP_THRESHOLD:
                    alerts.append(f"⚠ Sudden power drop detected on {daily_avg_ac.index[i]}")

        alert_text = "\n".join(alerts) if alerts else "No sudden power drops detected ✅"

        # Plot 1: Daily DC vs AC Power
        plt.figure(figsize=(12, 5))
        plt.plot(daily_dc_power.index, daily_dc_power.values, label='DC Power', marker='o', color='blue')
        plt.plot(daily_ac_power.index, daily_ac_power.values, label='AC Power', marker='s', color='green')
        plt.xlabel("Date")
        plt.ylabel("Power (W)")
        plt.title("Daily DC vs AC Power Generation")
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        power_fig = plt.gcf()
        plt.close()

        # Plot 2: Daily Yield
        plt.figure(figsize=(12, 5))
        plt.bar(daily_yield.index, daily_yield.values, color='orange', alpha=0.7)
        plt.xlabel("Date")
        plt.ylabel("Daily Yield (Wh)")
        plt.title("Daily Energy Yield")
        plt.xticks(rotation=45)
        plt.grid(True, axis='y')
        plt.tight_layout()
        yield_fig = plt.gcf()
        plt.close()

        # Plot 3: Hourly Power Pattern (First Day)
        sample_day = df['date'].iloc[0]
        day_df = df[df['date'] == sample_day]

        plt.figure(figsize=(12, 5))
        plt.plot(day_df['hour'], day_df['DC_POWER'], label='DC Power', marker='o', color='blue')
        plt.plot(day_df['hour'], day_df['AC_POWER'], label='AC Power', marker='s', color='green')
        plt.xlabel("Hour of Day")
        plt.ylabel("Power (W)")
        plt.title(f"Hourly Power Generation Pattern ({sample_day})")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        hourly_fig = plt.gcf()
        plt.close()

        # Summary Table
        summary_df = pd.DataFrame({
            "Date": daily_dc_power.index,
            "DC Power (W)": daily_dc_power.values,
            "AC Power (W)": daily_ac_power.values,
            "Daily Yield (Wh)": daily_yield.values,
            "Efficiency (%)": ((daily_ac_power / daily_dc_power) * 100).round(2).values
        })

        # Get plant info
        plant_ids = df['PLANT_ID'].unique()
        source_keys = df['SOURCE_KEY'].nunique()
        total_yield = df['TOTAL_YIELD'].max()

        system_report = f"""
📊 Conversion Efficiency: {conversion_efficiency:.2f}%
🔋 System Status: {system_status}
🏭 Plant ID(s): {', '.join(map(str, plant_ids))}
🔌 Number of Inverters: {source_keys}
⚡ Total Cumulative Yield: {total_yield:,.0f} Wh
📅 Data Period: {df['date'].min()} to {df['date'].max()}

⚠️ Alerts:
{alert_text}
        """

        return system_report, summary_df, power_fig, yield_fig, hourly_fig

    except Exception as e:
        return f"Error: {str(e)}", None, None, None, None

In [7]:
# ============================================================================
# 5. GRADIO INTERFACE - COMBINED TABS
# ============================================================================
with gr.Blocks(title="SAURNET Solar Monitoring System") as app:
    gr.Markdown("""
    # ☀️ SAURNET - Solar Grid Monitoring & Analysis System
    Complete solar panel monitoring with simulation, real data analysis, and ML capabilities
    """)

    with gr.Tabs():
        # TAB 1: Simulation
        with gr.Tab("🔄 System Simulation"):
            gr.Markdown("""
            ### Simulate 30-day solar system with battery and grid export
            **This generates random simulated data** - each run produces different results.
            Adjust panel capacity, battery size, and efficiency to see how the system performs.
            """)
            with gr.Row():
                panel_slider = gr.Slider(1, 10, value=5, label="Solar Panel Capacity (kW)")
                battery_slider = gr.Slider(5, 20, value=10, label="Battery Capacity (kWh)")
                efficiency_slider = gr.Slider(0.7, 1.0, value=0.85, label="System Efficiency")

            sim_btn = gr.Button("Run Simulation")
            sim_status = gr.Textbox(label="System Status")
            sim_table = gr.Dataframe(label="30-Day Energy Data")
            sim_plot = gr.Plot(label="Energy Flow Visualization")

            sim_btn.click(
                run_simulation,
                inputs=[panel_slider, battery_slider, efficiency_slider],
                outputs=[sim_table, sim_status, sim_plot]
            )
        # TAB 2: CSV Analysis
        with gr.Tab("📁 CSV Data Analysis"):
            gr.Markdown("""
            ### Upload YOUR OWN solar panel CSV data for analysis
            **This analyzes real data from your CSV file or URL** - not simulated data.

            Required columns: `DATE_TIME`, `PLANT_ID`, `SOURCE_KEY`, `DC_POWER`, `AC_POWER`, `DAILY_YIELD`, `TOTAL_YIELD`

            Example CSV format:
            ```
            DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
            15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
            15-05-2020 00:15,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
            ```
            """)

            with gr.Row():
                with gr.Column():
                    file_input = gr.File(label="Upload Solar Panel CSV Data")
                with gr.Column():
                    url_input = gr.Textbox(
                        label="Or Enter CSV URL",
                        placeholder="https://example.com/solar_data.csv",
                        lines=1
                    )

            gr.Markdown("*Provide either a file upload OR a URL (URL takes priority if both are provided)*")

            analyze_btn = gr.Button("Analyze System")

            report = gr.Textbox(label="System Analysis Report", lines=8)
            table = gr.Dataframe(label="Monthly Solar Data Summary")

            with gr.Row():
                plot1 = gr.Plot(label="Daily DC vs AC Power")
                plot2 = gr.Plot(label="Daily Energy Yield")
            plot3 = gr.Plot(label="Hourly Power Pattern (Sample Day)")

            analyze_btn.click(
                analyze_solar_data,
                inputs=[file_input, url_input],
                outputs=[report, table, plot1, plot2, plot3]
            )
        # TAB 3: ML Models
        with gr.Tab("🤖 ML Anomaly Detection"):
            gr.Markdown("""
            ### Train and test anomaly detection models
            **This generates synthetic sensor data** with fixed seed for reproducibility.
            Uses IsolationForest to detect anomalies in solar panel readings.
            """)

            samples_slider = gr.Slider(100, 500, value=200, step=50, label="Number of Samples")
            train_btn = gr.Button("Generate Data & Train Models")
            detect_btn = gr.Button("Detect Anomalies")

            ml_status = gr.Textbox(label="Status")
            ml_data = gr.Dataframe(label="Generated Data with Anomaly Flags")

            generated_data = gr.State()

            def train_pipeline(samples):
                data = generate_solar_data(int(samples))
                status = train_models(data)
                return status, data, data.head(10)

            def detect_pipeline(data):
                if data is None:
                    return "Please train models first", None
                result_data, status = detect_anomalies(data)
                return status, result_data

            train_btn.click(
                train_pipeline,
                inputs=samples_slider,
                outputs=[ml_status, generated_data, ml_data]
            )

            detect_btn.click(
                detect_pipeline,
                inputs=generated_data,
                outputs=[ml_status, ml_data]
            )
# Launch the app
app.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.
